# Pilot Test: Jupyter + Public LLM Memory Dump Feasibility

이 노트북은 사용자가 지정한 memory dump를 읽고, 로컬 분석 결과를 공개 LLM API에 연결하는 최소 동작 데모입니다.

In [1]:
from pathlib import Path
import json
import os
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DUMP_PATH = os.environ.get('DUMP_PATH', '/home/taejin/Jupyter/data/memory/memory.vmem').strip()
VMLINUX_PATH = os.environ.get('VMLINUX_PATH', '').strip()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DUMP_PATH =', DUMP_PATH)
print('VMLINUX_PATH =', VMLINUX_PATH or '(not provided)')
print('dump exists =', Path(DUMP_PATH).exists() if DUMP_PATH else False)
print('OPENAI_API_KEY configured =', bool(os.environ.get('OPENAI_API_KEY')))

PROJECT_ROOT = /home/taejin/Jupyter/jupyter-ramdump-analyzer
DUMP_PATH = /home/taejin/Jupyter/data/memory/memory.vmem
VMLINUX_PATH = (not provided)
dump exists = True
OPENAI_API_KEY configured = True


## Imports and runtime options

In [2]:
from analysis_pipeline import FeasibilityOptions, run_feasibility_analysis
from llm_assistant import LLMAssistant
from memory_kernel_analyzer import build_analysis_context, summarize_findings

MODEL = os.environ.get('OPENAI_MODEL', 'nvidia/nemotron-3-super-120b-a12b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')
ALLOW_RAW_CHUNK_EXCERPT = True
GENERATE_NEXT_STEPS = False

assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('MODEL =', MODEL)
print('API_BASE =', API_BASE)
assistant.is_configured()

MODEL = nvidia/nemotron-3-super-120b-a12b:free
API_BASE = https://openrouter.ai/api/v1


True

## Step 1. Build local analysis context

In [3]:
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 2. Inspect chunk samples sent to the LLM

In [4]:
for sample in context.raw_chunk_samples[:3]:
    print(sample)
    print('-' * 80)

{'offset': 0, 'size': 256, 'hex_preview': '53ff00f053ff00f0c3e200f053ff00f053ff00f054ff00f0888400f053ff00f0', 'ascii_preview': 'S...S.......S...S...T.......S...........V...V...V...V...W...........M...A.......9...Y...."J.....Y.......n...S...S.......'}
--------------------------------------------------------------------------------
{'offset': 536870912, 'size': 256, 'hex_preview': '280e3f001b0129001f0530001b0129001f0530001b0129001f0530001f053000', 'ascii_preview': '(.?...)...0...)...0...)...0...0...)...0...0...)...0...)...0...)...0...)...0...)...0...0...0...0...)...0...)...0...0...).'}
--------------------------------------------------------------------------------
{'offset': 1073741824, 'size': 256, 'hex_preview': 'fe86fe2124ed8f0525bda90b25ce981122bd0b1425fd8d1a252d872025ce8526', 'ascii_preview': '...!$...%...%..."...%...%-. %..&"(....)%../%|."..8%..>%..D%l.".x"..P%=.V%].\\%].b%..h%}.n%M.t%..z%...%.."...%...%.."...L.'}
-------------------------------------------------------------------

## Step 3. Run the feasibility pipeline

In [5]:
options = FeasibilityOptions(
    dump_path=DUMP_PATH,
    allow_raw_chunk_excerpt=ALLOW_RAW_CHUNK_EXCERPT,
    raw_chunk_limit=2,
    generate_next_steps=GENERATE_NEXT_STEPS,
)
report = run_feasibility_analysis(DUMP_PATH, assistant, options)
report.llm_enabled

True

## Step 4. Review local findings

In [6]:
print(json.dumps(report.findings, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 5. Review LLM output

API 키가 없으면 이 셀은 `None` 또는 오류 메시지를 보여줍니다.

In [7]:
print(report.llm_analysis)
print('\n' + '=' * 80 + '\n')
print(report.next_steps)


## 핵심 이상 징후 및 Root Cause 후보 요약

### 🚨 주요 이상 징후

1. **커널 패닉 증거**
   - `kernel_oops:BUG:` 패턴 2건 발견
   - `crashes:crash` 패턴 존재

2. **메모리 관리 체계 손상**
   - **"alloc magic is broken at %p: %lx"** - 메모리 할당자 내부 헤더 손상
   - **"out of memory"** - 메모리 부족 상황

3. **부트로더 단계 메모리 오류**
   - GRUB 메모리 함수 다수 발견: `grub_calloc`, `grub_malloc`, `grub_realloc`, `grub_zalloc`
   - **"PXE-E20: BIOS extended memory copy error"** - 네트워크 PXE 부팅 시 BIOS 메모리 복사 실패

### 🔍 Root Cause 후보

| 후보 | 증거 | 신뢰도 |
|------|------|--------|
| **메모리 할당자 손상** | alloc magic broken 메시지 | ⭐⭐⭐⭐ |
| **메모리 부족 (OOM)** | out of memory 문자열 | ⭐⭐⭐ |
| **PXE 부팅 메모리 오류** | PXE-E20 에러 + GRUB 함수 | ⭐⭐⭐ |
| **커널 버그/레이스 컨디션** | kernel_oops:BUG 패턴 | ⭐⭐ |

### ⚠️ 분석 제한사항
- **vmlinux 심볼 없이 문자열 기반 추정**
- **정밀한 구조체/레지스터 분석 미수행**
- **대용량 dump 샘플링만으로 패턴 파악**

> **결론**: 메모리 할당자 손상(alloc magic broken)이 가장 유력한 root cause이며, 이는 PXE 부팅 과정의 메모리 복사 오류나 커널 내부 레이스 컨디션으로 인한 것으로 추정됩니다.



None


## Step 6. Script generation example

In [8]:
if assistant.is_configured():
    script = assistant.generate_analysis_script(
        'memory dump에서 panic signal과 call trace 후보를 추출하는 Python 스크립트',
        findings=report.findings,
    )
    print(script)
else:
    print('OPENAI_API_KEY is not configured')


```python
#!/usr/bin/env python3
import re
from pathlib import Path

def extract_panic_signals(vmem_path):
    """Memory dump에서 panic signal과 call trace 후보 추출"""
    
    # Panic 관련 패턴 정의
    panic_patterns = [
        rb'kernel_oops:BUG:',
        rb'crashes:crash',
        rb'BUG:.*at',
        rb'Oops:.*',
        rb'panic:.*',
        rb'RIP:.*',
        rb'Call Trace:',
        rb'Stack trace:',
        rb'alloc magic is broken',
        rb'out of memory',
        rb'PXE-E20: BIOS extended memory copy error'
    ]
    
    # Call trace 후보 패턴 (함수명, 주소)
    calltrace_patterns = [
        rb'[a-z_][a-z0-9_]*\+0x[0-9a-f]+',  # 함수명+오프셋
        rb'\[<[0-9a-f]+>\]',  # 주소 괄호
        rb'grub_[a-z]+',  # grub 함수
        rb'__[a-z_]+',  # 커널 내부 함수
        rb'[a-z_]+_panic',  # panic 함수
        rb'[a-z_]+_error'  # error 함수
    ]
    
    results = {'panic_signals': [], 'calltrace_candidates': []}
    
    try:
        with open(vmem_path, 'rb') as f:
            # 파일 전체 읽기 (대용량이면 chunk로 나누

In [9]:
load_ext jupyter_ai

The jupyter_ai module is not an IPython extension.
